# Vectorize Full Law Data into a ChromaDB Knowledge Base (BGE Embeddings)

This notebook takes Thai land & building tax law data from an **Excel file** and:

1. Loads each law section as a `Document`.
2. Splits documents into smaller `Nodes`.
3. Embeds each node using a **HuggingFace BGE embedding model**.
4. Stores all embedded nodes into a **ChromaDB** persistent collection.

It is designed to be clean and publishable on GitHub alongside your AEI paper code.

> **Expected file structure (example):**
> - `notebooks/vectorize_full_law_chroma.ipynb` (this notebook)
> - `data/full_law_data.xlsx` (your law data; may be kept out of the repo or in `.gitignore`)


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install pandas chromadb llama-index-core llama-index-embeddings-huggingface llama-index-vector-stores-chroma

import pandas as pd
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from chromadb import PersistentClient
from llama_index.core.node_parser import SimpleNodeParser

print("✅ Imports loaded.")


## 1. Load Excel Data and Build Documents

We expect the Excel file to contain at least the columns:

- `Law_Name`: full name / citation of the provision  
- `Law_Detail`: the legal text

You can adjust the column names below if your schema differs.


In [ ]:
# === STEP 1: Load Excel Data ===
excel_path = "data/full_law_data.xlsx"  # adjust if your file is elsewhere
df = pd.read_excel(excel_path)

# Optional: Preview columns
print("📄 Columns in Excel:", df.columns.tolist())
print("Number of rows:", len(df))

# === STEP 2: Combine into string-based documents ===
# You can customize how to join fields below.
documents = []
for idx, row in df.iterrows():
    law_name = str(row.get("Law_Name", ""))
    law_detail = str(row.get("Law_Detail", ""))
    full_text = f"{law_name}\n{law_detail}".strip()
    metadata = {"FullName": law_name}
    documents.append(Document(text=full_text, metadata=metadata))

print(f"✅ Loaded {len(documents)} documents from Excel.")


## 2. Split into Nodes, Embed with BGE, and Store in ChromaDB

We now:

1. Use `SimpleNodeParser` to split each document into smaller chunks (`Nodes`).
2. Use a HuggingFace BGE model to embed each node.
3. Store embedded nodes in a persistent ChromaDB collection.


In [ ]:
# === STEP 3: Split into Nodes ===
parser = SimpleNodeParser()
nodes = parser.get_nodes_from_documents(documents)
print(f"✅ Parsed into {len(nodes)} nodes.")

# === STEP 4: Set up Embedding Model (BGE) ===
embed_model = HuggingFaceEmbedding(model_name="BAAI/BAAI/bge-m3")

# === STEP 5: Set up ChromaDB ===
db = PersistentClient(path="./chroma_db_full_law")
collection = db.get_or_create_collection(name="full_law_collection")
vector_store = ChromaVectorStore(chroma_collection=collection)

# === STEP 6: Add to Vector Store ===
# ChromaVectorStore in recent LlamaIndex versions uses `add` with nodes + embeddings.
# We compute embeddings node-by-node for clarity.
from tqdm import tqdm

texts = []
metadatas = []
ids = []

for node in tqdm(nodes, desc="Embedding & buffering nodes"):
    emb = embed_model.get_text_embedding(node.get_content(metadata_mode="all"))
    texts.append(node.get_content(metadata_mode="all"))
    metadatas.append(node.metadata or {})
    ids.append(node.node_id)

# Directly use the underlying Chroma collection for full control:
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
)

print("✅ All nodes embedded and stored in ChromaDB (collection: full_law_collection).")


## 3. Next Steps

Your law knowledge base is now in a persistent **ChromaDB** collection at `./chroma_db_full_law`.

You can later:

- Open the same collection and run similarity search.
- Wrap it in a LlamaIndex `VectorStoreIndex`.
- Use it inside your RAG pipeline for Thai land & building tax QA.

Example (in a separate script or notebook):

```python
from chromadb import PersistentClient
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

db = PersistentClient(path="./chroma_db_full_law")
collection = db.get_or_create_collection(name="full_law_collection")
vector_store = ChromaVectorStore(chroma_collection=collection)

index = VectorStoreIndex.from_vector_store(vector_store)

query_engine = index.as_query_engine()
response = query_engine.query("ที่ดินที่ใช้ประกอบเกษตรกรรมได้รับการยกเว้นภาษีอย่างไร?")
print(response)
```
